# BBO Function 1 — Part 2 reflection analysis

This notebook is for the **first 2D unknown function**.

It is designed to help answer the Part 2 reflection prompts:
- which evaluated inputs behaved like support vectors or boundary points;
- how surrogate gradients change with the inputs;
- how classification framing separates “good” and “bad” outputs;
- whether linear regression, SVM, or neural networks are most useful;
- which variables influence the surrogate most;
- whether the neural network captures nonlinear patterns better than simpler models.

Assumption: this BBO objective is being **minimised**.


## Version 2: classification framing — good vs bad outputs

Use this version for SVM/logistic regression/neural-network boundary questions.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load original function 1 data.
# Run this notebook from the same folder where initial_data/function_1 exists.
input_data = np.load('initial_data/function_1/initial_inputs.npy')
output_data = np.load('initial_data/function_1/initial_outputs.npy')

# Add your latest evaluated point here.
# You said the new input is (0.5, 0.5). Replace NEW_OUTPUT with the portal output.
NEW_POINT = np.array([[0.5, 0.5]])
NEW_OUTPUT = 2.6752879910742468e-9

if NEW_OUTPUT is not None:
    input_data = np.vstack([input_data, NEW_POINT])
    output_data = np.append(output_data, NEW_OUTPUT)

df = pd.DataFrame(input_data, columns=['x1', 'x2'])
df['y'] = output_data
df['log_abs_y'] = np.log(np.abs(output_data) + 1e-300)
df['rank_min'] = df['y'].rank(method='min', ascending=True).astype(int)

print(df.sort_values('y').to_string(index=True))
print("\\nBest point so far:")
print(df.loc[df['y'].idxmin()])


FileNotFoundError: [Errno 2] No such file or directory: 'initial_data/function_1/initial_inputs.npy'

In [ ]:

plt.figure(figsize=(7, 6))
sc = plt.scatter(df['x1'], df['x2'], c=df['log_abs_y'], s=100, edgecolors='black')
plt.colorbar(sc, label='log(|y|)')
for i, row in df.iterrows():
    plt.annotate(str(i), (row['x1'], row['x2']), xytext=(6, 6), textcoords='offset points')
plt.xlabel('x1')
plt.ylabel('x2')
plt.title('Observed function values for Function 1')
plt.grid(alpha=0.25)
plt.show()


In [ ]:

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import LeaveOneOut, cross_val_score

# For minimisation, define 'good' as the best 25% observed outputs.
# You can change GOOD_QUANTILE to 0.33 or 0.5 if there are too few good points.
GOOD_QUANTILE = 0.25
threshold = np.quantile(output_data, GOOD_QUANTILE)
labels = (output_data <= threshold).astype(int)

df_cls = df.copy()
df_cls['good_label'] = labels
df_cls['distance_to_threshold'] = np.abs(output_data - threshold)
print(f"Good/bad threshold: y <= {threshold:.6e}")
print(df_cls.sort_values('distance_to_threshold').to_string(index=True))

# Points with outputs close to the threshold are support-vector-like:
# they are near the boundary between 'good' and 'bad'.
support_like = df_cls.sort_values('distance_to_threshold').head(min(5, len(df_cls)))
print("\\nSupport-vector-like observed points:")
print(support_like[['x1','x2','y','good_label','distance_to_threshold']])


In [ ]:

X = input_data
y_class = labels

models = {
    "logistic_regression": make_pipeline(StandardScaler(), LogisticRegression()),
    "linear_svm": make_pipeline(StandardScaler(), SVC(kernel='linear', probability=True)),
    "rbf_svm": make_pipeline(StandardScaler(), SVC(kernel='rbf', gamma='scale', probability=True)),
}

for name, model in models.items():
    model.fit(X, y_class)
    if len(np.unique(y_class)) == 2 and min(np.bincount(y_class)) >= 2:
        scores = cross_val_score(model, X, y_class, cv=LeaveOneOut(), scoring='accuracy')
        print(name, "LOO accuracy:", scores.mean())
    else:
        print(name, "fitted. Cross-validation skipped because one class has too few points.")


In [ ]:

# Boundary visualisation.
grid_n = 300
xx, yy = np.meshgrid(np.linspace(0, 1, grid_n), np.linspace(0, 1, grid_n))
grid = np.column_stack([xx.ravel(), yy.ravel()])

for name, model in models.items():
    if hasattr(model[-1], "predict_proba"):
        score = model.predict_proba(grid)[:, 1]
    else:
        score = model.decision_function(grid)
    score_grid = score.reshape(grid_n, grid_n)

    plt.figure(figsize=(7, 6))
    cs = plt.contourf(xx, yy, score_grid, levels=30, alpha=0.85)
    plt.colorbar(cs, label='P(good) or decision score')
    plt.contour(xx, yy, score_grid, levels=[0.5] if score_grid.min() <= 0.5 <= score_grid.max() else 10, colors='black')
    plt.scatter(X[:,0], X[:,1], c=y_class, edgecolors='black', s=90)
    for i, (a,b) in enumerate(X):
        plt.annotate(str(i), (a,b), xytext=(5,5), textcoords='offset points')
    plt.xlabel('x1'); plt.ylabel('x2')
    plt.title(f'Good/bad boundary: {name}')
    plt.show()


## Reflection evidence from this version

Use this notebook to say:

I reframed the BBO task as a binary classification problem by labelling the lowest-output observations as “good” and the rest as “bad”. The support-vector-like observations are the points closest to this threshold, because small changes in their output would move them from one class to the other. A linear classifier is easier to interpret, but it may miss curved boundaries. The RBF SVM is more flexible and better suited if the good region is local or nonlinear, but it is also easier to overfit because the data set is still small.